In [1]:
import os, time, socket
import numpy as np, pandas as pd, torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(socket.gethostname(), device, torch.cuda.get_device_name(0) if device == "cuda" else "")

OUT = "results/rebuttal_radical"; os.makedirs(OUT, exist_ok=True)
F_RAD = f"{OUT}/radical.csv"
DIMS = [10, 20, 30]
POOLS = ['Continuous Only', 'Discrete Only', 'Strictly Super-Gaussian', 'Zero Gaussian', 'Full Hybrid']
TRIALS = range(10)
N = 10_000


gpux2 cuda NVIDIA GeForce RTX 2080 Ti


In [2]:
# =============================================================================
# Amari performance index
# =============================================================================
def amari_error(W, A):
    """
    W : estimated unmixing (d×d)
    A : true mixing (d×d)
    Returns scalar ∈ [0, 1]; 0 = perfect separation.
    """
    if W is None or np.any(np.isnan(W)):
        return np.nan
    P = np.abs(W @ A)
    n = P.shape[0]
    # Sum each normalised row/column first, THEN subtract 1 once per row/column.
    # (Subtracting 1 inside np.sum would apply it to every element — d× too many.)
    row_terms = np.sum(P / np.max(P, axis=1, keepdims=True), axis=1) - 1.0
    col_terms = np.sum(P / np.max(P, axis=0, keepdims=True), axis=0) - 1.0
    return (np.sum(row_terms) + np.sum(col_terms)) / (2.0 * n)

In [3]:
# =============================================================================
# Data generator (exact port of ablation notebook)
# =============================================================================
def generate_mixture(n_dim, n_samples, config_type, seed=None):
    """
    config_type ∈ {'Continuous Only', 'Discrete Only',
                   'Strictly Super-Gaussian', 'Zero Gaussian', 'Full Hybrid'}.
    """
    if seed is not None:
        np.random.seed(seed)

    def gen_laplace():     return np.random.laplace(0, 1/np.sqrt(2), n_samples)
    def gen_bernoulli():   return np.random.choice([-1., 1.], n_samples)
    def gen_uniform():     return np.random.uniform(-np.sqrt(3), np.sqrt(3), n_samples)
    def gen_student_t():   s = np.random.standard_t(3, n_samples); return s / np.std(s)
    def gen_poisson():     s = np.random.poisson(3, n_samples); return (s - s.mean()) / s.std()
    def gen_binomial():    s = np.random.binomial(10, .5, n_samples); return (s - s.mean()) / s.std()
    def gen_chisquare():   s = np.random.chisquare(2, n_samples); return (s - s.mean()) / s.std()
    def gen_exponential(): s = np.random.exponential(1, n_samples); return (s - s.mean()) / s.std()

    pools = {
        'Full Hybrid':             [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t,
                                    gen_poisson, gen_binomial, gen_chisquare, gen_exponential],
        'Continuous Only':         [gen_laplace, gen_uniform, gen_student_t,
                                    gen_chisquare, gen_exponential],
        'Discrete Only':           [gen_bernoulli, gen_poisson, gen_binomial],
        'Strictly Super-Gaussian': [gen_laplace, gen_student_t, gen_chisquare, gen_exponential],
        'Zero Gaussian':           [gen_laplace, gen_bernoulli, gen_uniform, gen_student_t,
                                    gen_poisson, gen_binomial, gen_chisquare, gen_exponential],
        # Pure single-distribution configs (no Gaussian component)
        'Pure Laplace':            [gen_laplace],
        'Pure Uniform':            [gen_uniform],
        'Pure Student-t':          [gen_student_t],
        'Pure Bernoulli':          [gen_bernoulli],
        'Pure Poisson':            [gen_poisson],
        'Pure Binomial':           [gen_binomial],
        'Pure Chi-square':         [gen_chisquare],
        'Pure Exponential':        [gen_exponential],
    }

    active_pool = pools[config_type]
    sources = []
    is_pure = config_type == 'Zero Gaussian' or config_type.startswith('Pure ')
    if not is_pure:
        sources.append(np.random.normal(0, 1, n_samples))
        n_to_gen = n_dim - 1
    else:
        n_to_gen = n_dim

    for _ in range(n_to_gen):
        sources.append(np.random.choice(active_pool)())

    S = np.stack(sources)
    np.random.shuffle(S)

    cond_num = 1000.0
    while cond_num > 100:
        A = np.random.randn(n_dim, n_dim)
        cond_num = np.linalg.cond(A)
    return A @ S, A

In [4]:
def radical(X, K=150, R=30, sigma=0.175, seed=0):
    """RADICAL (Learned-Miller & Fisher, JMLR 2003). Whiten, then d-1 Jacobi sweeps over all
    pairs (i, j). Each pair is rotated by the angle in [-pi/4, pi/4] (K grid points) that minimizes
    the sum of the two m-spacing entropy estimates (m = sqrt(N)), computed on R noisy replicates
    of the pair (noise std sigma). Returns the total unmixing matrix W (rotation @ whitening)."""
    X = X - X.mean(1, keepdims=True)
    evals, E = np.linalg.eigh(np.cov(X))
    W_white = (E / np.sqrt(evals)).T
    d, n = X.shape
    m = int(np.sqrt(n))
    g = torch.Generator(device=device).manual_seed(seed)
    Z = torch.tensor(W_white @ X, dtype=torch.float32, device=device)
    th = torch.linspace(-np.pi / 4, np.pi / 4, K, device=device)
    c, s = torch.cos(th)[:, None], torch.sin(th)[:, None]
    Rot = np.eye(d)

    def spacing_entropy(Y):   # (K, n*R) -> (K,); additive constants dropped as in the original code
        Ys = torch.sort(Y, dim=1).values
        return torch.log(Ys[:, m:] - Ys[:, :-m]).sum(1)

    for _ in range(d - 1):
        for i in range(d - 1):
            for j in range(i + 1, d):
                a, b = Z[[i, j]].repeat(1, R) + sigma * torch.randn(2, n * R, generator=g, device=device)
                H = spacing_entropy(c * a - s * b) + spacing_entropy(s * a + c * b)
                t = th[int(torch.argmin(H))].item()
                G = np.array([[np.cos(t), -np.sin(t)], [np.sin(t), np.cos(t)]])
                Z[[i, j]] = torch.tensor(G, dtype=torch.float32, device=device) @ Z[[i, j]]
                Rot[[i, j]] = G @ Rot[[i, j]]
    return Rot @ W_white

# sanity check (~2 s): d=4 mixture, normalized Amari should be ~0.01 or below
X, A = generate_mixture(4, N, 'Full Hybrid', seed=0)
print("sanity, normalized Amari:", round(amari_error(radical(X), A) / 3, 4))


sanity, normalized Amari: 0.0109


In [5]:
def append_rows(path, rows):
    pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

done = set(map(tuple, pd.read_csv(F_RAD)[["Dim", "Config", "Trial"]].itertuples(index=False))) if os.path.exists(F_RAD) else set()
for dim in DIMS:
    for trial in TRIALS:                  # trial outside config: a partial d=30 still covers every regime
        for config in POOLS:
            if (dim, config, trial) in done: continue
            X, A = generate_mixture(dim, N, config, seed=trial)
            t0 = time.perf_counter()
            W = radical(X, seed=trial)
            row = {"Dim": dim, "Config": config, "Trial": trial, "Method": "RADICAL",
                   "Amari": amari_error(W, A), "Time": time.perf_counter() - t0, "Device": device}
            append_rows(F_RAD, [row])
            with open(f"{OUT}/progress.log", "a") as fh:
                fh.write(f"{time.strftime('%H:%M:%S')} d={dim} {config} trial={trial} "
                         f"amari_norm={row['Amari'] / (dim - 1):.4f} t={row['Time']:.0f}s\n")
print("RADICAL done")


RADICAL done
